In [31]:
# Import everything at the top of your notebook
import pandas as pd
import numpy as np
import re
from dateutil import parser as dateparser
import pytz
import warnings
warnings.filterwarnings('ignore')

# Load the raw dataset
df = pd.read_csv('Cloud Raw dataset.csv') 

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (10200, 28)
Columns: ['Usage_ID', 'Account', 'TS', 'Service', 'SKU', 'Usage', 'Unit', 'Cost', 'Currency', 'Region', 'Free_Tier_Flag', 'Tag_Owner', 'Tag_Env', 'Resource_ID', 'Ticket_ID', 'Ticket_Text', 'Incident_ID', 'Price_Version', 'Pricing_Type', 'Department', 'Project', 'SLA_Event', 'Log_Skew_Seconds', 'FX_Rate', 'Purchase_Type', 'Anomaly_Flag', 'Duplicate_Flag', 'Cost_Allocation_Valid']


,Usage_ID,Account,TS,Service,SKU,Usage,Unit,Cost,Currency,Region,...,Pricing_Type,Department,Project,SLA_Event,Log_Skew_Seconds,FX_Rate,Purchase_Type,Anomaly_Flag,Duplicate_Flag,Cost_Allocation_Valid
0,U23168,acct-025,2025-06-10 04:15:00-08:00,Database,VM.PREM.4,271973,min,534.55,INR,eu-central-1,...,reserved,prod,NaN,True,-393,104.6,SPOT,NaN,NaN,NaN
1,U29497,ACCT-019,21-09-2025 19:45:00,Security,Vm_prem_8,269223,hour,2743.69,USD,EU-CENTRAL-1,...,Spot,ds,DELTA,true,-275,88.53,SPOT,NaN,NaN,NaN
2,U29557,acct041,2025-09-29T19:38:00+05:30,Containers,VM.prem.4,75327,minute,€8935.89,Us Dollar,Canada Central,...,ri,eng,Epsilon,yes,-354,na,SPOT,NaN,NaN,NaN
3,U22314,ACCT-037,2025-02-04 12:35:00,Database,vm-PREM-4,"222,401",hours,$7529.96,NaN,ap_southeast_1,...,Reserved,product,proj-epsilon,No,100,88.49,ON_DEMAND,NaN,NaN,NaN
4,U22710,ACCT-023,2026-02-05T14:34:00+05:30,Compute,vmPREM16,243705,hrs,"₹7,347.25",NaN,us_east_1,...,SPOT,unknown,NaN,FALSE,-155,NaN,SPOT,NaN,NaN,NaN


In [32]:
# 1. Basic shape
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# 2. Missing values per column
print(" MISSING VALUES\n")
print(df.isnull().sum().sort_values(ascending=False))

# 3. Unique values in key columns
print("ACCOUNT VARIANTS\n")
print(df['Account'].value_counts().head(20))

print("UNIT VARIANTS\n")
print(df['Unit'].value_counts())

print("REGION VARIANTS \n")
print(df['Region'].value_counts())

print("PRICING TYPE VARIANTS\n")
print(df['Pricing_Type'].value_counts())

# 4. Sample the messy rows
print("SAMPLE MESSY TIMESTAMPS\n")
print(df['TS'].sample(10).values)

Total rows: 10200
Total columns: 28
 MISSING VALUES

Anomaly_Flag             10200
Duplicate_Flag           10200
Cost_Allocation_Valid    10200
FX_Rate                   3411
Incident_ID               2911
Free_Tier_Flag            2361
Ticket_ID                 2304
Tag_Owner                 2113
Currency                  1857
Resource_ID               1495
Price_Version             1261
Tag_Env                   1053
Pricing_Type              1048
Project                    876
SLA_Event                  537
Department                 324
Region                       0
Account                      0
TS                           0
Service                      0
Usage_ID                     0
Usage                        0
Unit                         0
Cost                         0
SKU                          0
Ticket_Text                  0
Log_Skew_Seconds             0
Purchase_Type                0
dtype: int64
ACCOUNT VARIANTS

Account
Acct-048        40
acct-0014       40
AC

## Scenario-1

In [33]:
def clean_account_id(val):
    if pd.isna(val) or str(val).strip() in ('', 'N/A', 'na', 'NA'):
        return None
    val = str(val).strip().upper()          # strip spaces + uppercase
    val = re.sub(r'[\s_]', '-', val)        # replace space/underscore with hyphen
    # normalize: ACCT003 → ACCT-003
    val = re.sub(r'(ACCT)(\d+)', r'\1-\2', val)
    return val

df['Account_Clean'] = df['Account'].apply(clean_account_id)


In [34]:
def clean_ticket_id(val):
    if pd.isna(val) or str(val).strip() in ('', 'N/A', 'na', 'NA'):
        return None
    val = str(val).strip().upper()
    # normalize: tkt-6679 / ticket6679 / TICKET-6679 → T-6679
    val = re.sub(r'^(TICKET|TKT)-?(\d+)$', r'T-\2', val)
    return val

df['Ticket_ID_Clean'] = df['Ticket_ID'].apply(clean_ticket_id)

print(df[['Account', 'Account_Clean', 'Ticket_ID', 'Ticket_ID_Clean']].head(10))

        Account Account_Clean    Ticket_ID Ticket_ID_Clean
0      acct-025      ACCT-025  TICKET-6428          T-6428
1    ACCT-019        ACCT-019       t-6645          T-6645
2       acct041      ACCT-041     tkt-6879          T-6879
3    ACCT-037        ACCT-037       t-2840          T-2840
4      ACCT-023      ACCT-023       T-4353          T-4353
5       ACCT009      ACCT-009     TKT-8389          T-8389
6       acct044      ACCT-044          NaN             NaN
7    ACCT-042        ACCT-042          NaN             NaN
8      Acct-047      ACCT-047       T-8015          T-8015
9      acct-046      ACCT-046          NaN             NaN


## Scenario-2

In [35]:
def clean_timestamp(val):
    if pd.isna(val) or str(val).strip() in ('', 'N/A', 'na'):
        return None
    val = str(val).strip()
    
    # Fix invalid hours (e.g., 25:05 → 01:05)
    val = re.sub(r' (2[4-9]|[3-9]\d):(\d{2})', 
                  lambda m: f" 0{int(m.group(1))-24}:{m.group(2)}", val)
    
    # Fix slash separators → hyphens
    val = re.sub(r'(\d{4})/(\d{2})/(\d{2})', r'\1-\2-\3', val)
    
    # Fix DD-MM-YYYY → YYYY-MM-DD
    val = re.sub(r'^(\d{2})-(\d{2})-(\d{4})', r'\3-\2-\1', val)
    
    try:
        dt = dateparser.parse(val)              # parse whatever format remains
        if dt.tzinfo is None:
            dt = pytz.utc.localize(dt)          # assume UTC if no timezone
        else:
            dt = dt.astimezone(pytz.utc)        # convert to UTC
        return dt.strftime('%Y-%m-%d %H:%M:%S+00:00')
    except:
        return None                             # flag as unparseable

df['TS_UTC'] = df['TS'].apply(clean_timestamp)
df['TS_Parse_Failed'] = df['TS_UTC'].isna() & df['TS'].notna()

print(f"Successfully parsed: {df['TS_UTC'].notna().sum()}")
print(f"Failed to parse: {df['TS_Parse_Failed'].sum()}")

Successfully parsed: 10200
Failed to parse: 0


## Scenario-3

In [36]:
SKU_CATALOG = {
    'VM-STD-2': ['vm-std-2','VM_STD_2','vm.std.2','vmstd2','VM-STD-2'],
    'VM-STD-4': ['vm-std-4','VM_STD_4','vm.std.4','vmstd4','VM-STD-4'],
    'VM-STD-8': ['vm-std-8','VM_STD_8','vm.std.8','vmstd8'],
    'VM-PREM-4': ['vm-prem-4','VM_PREM_4','vmprem4','VM-PREM-4'],
    'VM-BASIC-2': ['vm-basic-2','VM_BASIC_2','vmbasic2'],
    # ... add all SKUs
}

sku_lookup = {}
for canonical, variants in SKU_CATALOG.items():
    for v in variants:
        sku_lookup[v.upper().replace('_','-').replace('.','-')] = canonical

def clean_sku(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    normalized = str(val).strip().upper()
    normalized = re.sub(r'[_\.]', '-', normalized)   # unify separators
    return sku_lookup.get(normalized, normalized)     # return canonical or best guess

df['SKU_Clean'] = df['SKU'].apply(clean_sku)

print(df[['SKU', 'SKU_Clean']].drop_duplicates().head(20))


            SKU    SKU_Clean
0     VM.PREM.4    VM-PREM-4
1     Vm_prem_8    VM-PREM-8
2     VM.prem.4    VM-PREM-4
3     vm-PREM-4    VM-PREM-4
4      vmPREM16     VMPREM16
5    VM-BASIC-8   VM-BASIC-8
6    vm.prem.16   VM-PREM-16
7    vm_prem_16   VM-PREM-16
8   vm_basic_16  VM-BASIC-16
9    vm.basic.2   VM-BASIC-2
10    VM-prem-8    VM-PREM-8
11     VMBASIC8     VMBASIC8
12      vmprem2      VMPREM2
13       VMStd2     VM-STD-2
14     VM-STD-2     VM-STD-2
15    VM-PREM-2    VM-PREM-2
16     vmbasic4     VMBASIC4
17    VM-std-16    VM-STD-16
18      Vmprem2      VMPREM2
19    Vm.PREM.8    VM-PREM-8


# Scenario-4

In [37]:
Conversion_of_Units = {
    'sec': 1, 'second': 1, 'seconds': 1, 's': 1,
    'min': 60, 'minute': 60, 'minutes': 60,
    'hr': 3600, 'hour': 3600, 'hours': 3600, 'hrs': 3600,
}

def clean_usage_col(usage_val, unit_val):
    try:
        # Remove commas from numbers like 12,000
        usage_string = str(usage_val).replace(',', '').strip()
        usage_numeric = float(usage_string)
    except:
        return None
    
    unit = str(unit_val).strip().lower()
    multiplier = Conversion_of_Units.get(unit, 1)   # default to 1 if unit unknown
    return int(usage_numeric * multiplier)

df['Usage_Seconds'] = df.apply(
    lambda row: clean_usage_col(row['Usage'], row['Unit']), axis=1
)

print(df[['Usage', 'Unit', 'Usage_Seconds']].head(15))
print(f"\nNull usage records: {df['Usage_Seconds'].isna().sum()}")

      Usage     Unit  Usage_Seconds
0    271973      min       16318380
1    269223     hour      969202800
2     75327   minute        4519620
3   222,401    hours      800643600
4    243705      hrs      877338000
5    176226       hr      634413600
6    246185      sec         246185
7    105445      min        6326700
8    323952      hrs     1166227200
9   274,121      sec         274121
10   128568        s         128568
11   323711    hours     1165359600
12   182172  minutes       10930320
13    14280        s          14280
14   271209      min       16272540

Null usage records: 0


# Scenario-5

In [38]:
def clean_cost_col(val):
    if pd.isna(val) or str(val).strip() in ('', 'N/A', 'na'):
        return None
    val = str(val)
    # Remove all currency symbols and letters
    val = re.sub(r'[₹$€£¥A-Za-z\s]', '', val)
    # Remove comma thousands separators
    # But keep decimal point — handle European format (1.234,56)
    if ',' in val and '.' in val:
        if val.index(',') < val.index('.'):
            val = val.replace(',', '')          # 1,234.56 → 1234.56
        else:
            val = val.replace('.', '').replace(',', '.')  # 1.234,56 → 1234.56
    elif ',' in val:
        val = val.replace(',', '.')             # 1234,56 → 1234.56
    try:
        return round(float(val), 2)
    except:
        return None

def cleaning_currency(val):
    if pd.isna(val) or str(val).strip() in ('', 'N/A', 'na', 'NA'):
        return 'UNKNOWN'
    val = str(val).strip().upper()
    if any(x in val for x in ['USD', 'US DOLLAR', 'DOLLAR', '$']):
        return 'USD'
    if any(x in val for x in ['INR', '₹', 'RUPEE']):
        return 'INR'
    if any(x in val for x in ['EUR', '€', 'EURO']):
        return 'EUR'
    return val

df['Cost_Clean']     = df['Cost'].apply(clean_cost_col)
df['Currency_Clean'] = df['Currency'].apply(cleaning_currency)

print(df[['Cost', 'Currency', 'Cost_Clean', 'Currency_Clean']].head(15))

           Cost   Currency  Cost_Clean Currency_Clean
0        534.55        INR      534.55            INR
1       2743.69        USD     2743.69            USD
2      €8935.89  Us Dollar     8935.89            USD
3      $7529.96        NaN     7529.96        UNKNOWN
4     ₹7,347.25        NaN     7347.25        UNKNOWN
5     €4,234.88        usd     4234.88            USD
6     €4,015.03        NaN     4015.03        UNKNOWN
7      ₹1671.73  Us Dollar     1671.73            USD
8   USD 8049.73        eur     8049.73            EUR
9     ₹6,702.70  Us Dollar     6702.70            USD
10     €7815.52        eur     7815.52            EUR
11   INR2510.32        EUR     2510.32            EUR
12   INR6266.52        usd     6266.52            USD
13   INR4238.43        NaN     4238.43        UNKNOWN
14   INR5854.70        EUR     5854.70            EUR


# Scenario-6

In [39]:
REGION_MAPPING = {
    # ap-south-1 variants
    'AP-SOUTH-1':        'ap-south-1',
    'AP_SOUTH_1':        'ap-south-1',
    'AP SOUTH 1':        'ap-south-1',
    'AP-SOUTH1':         'ap-south-1',
    'ASIA PACIFIC (MUMBAI)': 'ap-south-1',
    # us-east-1 variants
    'US-EAST-1':         'us-east-1',
    'US_EAST_1':         'us-east-1',
    'US EAST 1':         'us-east-1',
    'USEAST1':           'us-east-1',
    'UNITED STATES EAST':'us-east-1',
    # eu-central-1 variants
    'EU-CENTRAL-1':      'eu-central-1',
    'EU_CENTRAL_1':      'eu-central-1',
    'EUCENTRAL1':        'eu-central-1',
    'EU CENTRAL':        'eu-central-1',
    'EU-CENTRAL1':       'eu-central-1',
    # ca-central-1 variants
    'CA-CENTRAL-1':      'ca-central-1',
    'CA-CENTRAL':        'ca-central-1',
    'CA_CENTRAL_1':      'ca-central-1',
    'CANADA CENTRAL':    'ca-central-1',
    'CA CENTRAL 1':      'ca-central-1',
    # ap-southeast-1 variants
    'AP-SOUTHEAST-1':    'ap-southeast-1',
    'AP_SOUTHEAST_1':    'ap-southeast-1',
    'AP-SE-1':           'ap-southeast-1',
    'SINGAPORE':         'ap-southeast-1',
}

def cleaning_region(val):
    if pd.isna(val) or str(val).strip() in ('', 'N/A', 'na'):
        return None
    normalize = str(val).strip().upper()
    normalize = re.sub(r'\s+', ' ', normalize)
    return REGION_MAPPING.get(normalize, normalize.lower())

df['Region_Clean'] = df['Region'].apply(cleaning_region)

print(df[['Region', 'Region_Clean']].value_counts().head(20))

Region                 Region_Clean  
ap-southeast-1         ap-southeast-1    435
Canada Central         ca-central-1      412
ap-se-1                ap-southeast-1    411
ap_southeast_1         ap-southeast-1    410
ca-central-1           ca-central-1      407
ca central 1           ca-central-1      404
Singapore              ap-southeast-1    404
AP-SE-1                ap-southeast-1    389
ca_central_1           ca-central-1      382
CA-CENTRAL             ca-central-1      381
US-EAST-1              us-east-1         372
Asia Pacific (Mumbai)  ap-south-1        358
ap-south-1             ap-south-1        355
EU Central             eu-central-1      351
USEast1                us-east-1         351
eu-central-1           eu-central-1      350
eu_central_1           eu-central-1      345
ap south 1             ap-south-1        343
us-east-1              us-east-1         340
us east 1              us-east-1         339
Name: count, dtype: int64
